# 02 - Brand Specification Generator

##  Objective
This notebook demonstrates how to transform a raw, unstructured brand brief into a structured **Brand Specification** using a Large Language Model (LLM).

The Brand Specification is the **foundation** of the entire BRANDORA pipeline. Every subsequent step (name generation, slogan creation, color palette selection, logo generation) depends on this structured data.

---

##  What This Notebook Does

1. **Loads** a sample brand brief from `test_briefs.json`
2. **Connects** to Groq API (using secure environment variables)
3. **Sends** the brief to an LLM with a carefully crafted prompt
4. **Forces** the LLM to output structured JSON (not free-form text)
5. **Validates** the output to ensure it matches the expected schema

---

##  Key Technical Decisions

### Why Groq?
- Free tier with generous rate limits
- Fast inference (Llama 3.3 70B runs in <1 second)
- Supports JSON mode for structured outputs

### Why JSON Mode?
Without JSON mode, the LLM might output:

In [1]:
import os
import json
from openai import OpenAI
from kaggle_secrets import UserSecretsClient

# 1. قراءة المفاتيح السرية من Kaggle Secrets
user_secrets = UserSecretsClient()
openrouter_key = user_secrets.get_secret("OPENROUTER_API_KEY")
inception_key = user_secrets.get_secret("INCEPTION_API_KEY")

# 2. تهيئة عميل OpenRouter (للنماذج الرخيصة)
openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=openrouter_key,
)

# 3. تهيئة عميل Inception Labs (لنموذج Mercury)
inception_client = OpenAI(
    base_url="https://api.inceptionlabs.ai/v1",
    api_key=inception_key,
)

print("✅ تم الاتصال بالمنصتين بنجاح!")
print("📡 OpenRouter: جاهز للنماذج الرخيصة")
print("📡 Inception Labs: جاهز لـ Mercury")

✅ تم الاتصال بالمنصتين بنجاح!
📡 OpenRouter: جاهز للنماذج الرخيصة
📡 Inception Labs: جاهز لـ Mercury


In [4]:
!pip install --upgrade torchaudio -q
print("✅ تم تحديث torchaudio ليتوافق مع PyTorch!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 60.4 MB/s eta 0:00:00
✅ تم تحديث torchaudio ليتوافق مع PyTorch!


In [ ]:


try:
    response = inception_client.chat.completions.create(
        model="mercury-2.5",
        messages=[
            {"role": "user", "content": "Say hello in one sentence."}
        ],
        max_tokens=500,
        reasoning_effort="low"
    )
    
    content = response.choices[0].message.content
    
    if content:
        print(f"✅ نجح Mercury! الرد: '{content.strip()}'")
        print(f"\n📊 إحصائيات الرد:")
        print(f"   - finish_reason: {response.choices[0].finish_reason}")
        if hasattr(response, 'usage') and response.usage:
            print(f"   - prompt_tokens: {response.usage.prompt_tokens}")
            print(f"   - completion_tokens: {response.usage.completion_tokens}")
            print(f"   - total_tokens: {response.usage.total_tokens}")
    else:
        print("⚠️ المحتوى لا يزال فارغاً")
        print(f"   - finish_reason: {response.choices[0].finish_reason}")
        if hasattr(response, 'usage') and response.usage:
            print(f"   - total_tokens المستهلكة: {response.usage.total_tokens}")
    
except Exception as e:
    print(f"❌ فشل! السبب: {str(e)[:200]}")

In [ ]:
import requests, json, re, time, gc
import pandas as pd

# تحميل البريفات (نفس ما فعلنا في الكود السابق)
url = "https://raw.githubusercontent.com/maram-elaian/brandora/main/data/test_briefs.json"
briefs = requests.get(url).json()
print(f"📋 عدد البريفات: {len(briefs)}\n")

def brief_to_prompt(b):
    return (
        f"Generate a brand name and a short tagline for this business.\n"
        f"Industry: {b['industry']}\n"
        f"Target audience: {b['target_audience']}\n"
        f"Brand purpose: {b['brand_purpose']}\n"
        f"Personality: {', '.join(b['personality'])}\n"
        f"Tone: {b['tone']}\n"
        f"Reply ONLY in this exact format: Name: ... | Tagline: ..."
    )

def clean_output(text):
    # إزالة أي تفكير داخلي أو تنسيق JSON
    if not text:
        return ""
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()
    text = re.sub(r'```json\s*', '', text)
    text = re.sub(r'```\s*', '', text)
    return text.strip()

# إعداد عميل Inception Labs
from openai import OpenAI
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
inception_key = user_secrets.get_secret("INCEPTION_API_KEY")
inception_client = OpenAI(
    base_url="https://api.inceptionlabs.ai/v1",
    api_key=inception_key
)

mercury_results = []
failed_briefs = []

print(f"⏳ بدء اختبار Mercury 2.5 على {len(briefs)} بريف...\n")

for i, brief in enumerate(briefs):
    try:
        messages = [
            {
                "role": "system",
                "content": "You are a branding expert. Generate a brand name and tagline."
            },
            {
                "role": "user",
                "content": brief_to_prompt(brief)
            }
        ]

        # استدعاء Mercury عبر Inception API
        response = inception_client.chat.completions.create(
            model="mercury-2.5",
            messages=messages,
            max_tokens=500,  # Mercury يحتاج توكنات أكثر بسبب Diffusion-based
            reasoning_effort="low"  # لتقليل التفكير الداخلي
        )

        # استخراج النتيجة
        raw_output = response.choices[0].message.content
        
        if raw_output:
            clean = clean_output(raw_output)
            mercury_results.append({
                "brief_id": brief["id"],
                "industry": brief["industry"],
                "model": "Mercury-2.5",
                "output": clean
            })
            print(f"   [{i+1}/{len(briefs)}] {brief['id']} ✅")
        else:
            print(f"   [{i+1}/{len(briefs)}] {brief['id']} ⚠️ محتوى فارغ")
            failed_briefs.append(brief["id"])

    except Exception as e:
        print(f"   [{i+1}/{len(briefs)}] {brief['id']} ❌ خطأ: {str(e)[:80]}")
        failed_briefs.append(brief["id"])

    # تأخير بسيط لتجنب Rate Limiting
    time.sleep(0.5)

print(f"\n✅ نجح: {len(mercury_results)}/{len(briefs)}")
print(f"❌ فشل: {len(failed_briefs)}/{len(briefs)}")

# حفظ النتائج
df_mercury = pd.DataFrame(mercury_results)
df_mercury.to_csv("mercury_results.csv", index=False)
print("\n💾 النتائج محفوظة بـ mercury_results.csv")
df_mercury.head(10)

In [5]:
pip install -U bitsandbytes>=0.46.1

Note: you may need to restart the kernel to use updated packages.


In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
import traceback
import re
import gc

print(f"GPU: {torch.cuda.get_device_name(0)}")

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

models_to_check = [
    {"name": "Qwen/Qwen3-8B", "model_id": "Qwen/Qwen3-8B"},
    {"name": "LiquidAI/LFM2.5-2.6B", "model_id": "LiquidAI/LFM2.5-2.6B"},
]

results = {}  # عشان نخزن الردود ونقارنها بعدين

print("🔍 بدء اختبار النماذج محلياً على GPU...\n")

for model_info in models_to_check:
    print(f"⏳ جاري تحميل: {model_info['name']}")
    model = None  # نعرّفه هون عشان الـ finally يلاقيه حتى لو فشل التحميل بنفسه
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_info["model_id"])
        model = AutoModelForCausalLM.from_pretrained(
            model_info["model_id"],
            quantization_config=quant_config,
            device_map="auto"
        )

        messages = [{"role": "user", "content": "Generate a brand name and a short tagline for a modern coffee shop targeting university students. Reply in this format: Name: ... | Tagline: ..."}]

        try:
            inputs = tokenizer.apply_chat_template(
                messages, add_generation_prompt=True,
                return_tensors="pt", return_dict=True,
                enable_thinking=False
            ).to(model.device)
        except TypeError:
            inputs = tokenizer.apply_chat_template(
                messages, add_generation_prompt=True,
                return_tensors="pt", return_dict=True
            ).to(model.device)

        outputs = model.generate(**inputs, max_new_tokens=300)
        input_length = inputs["input_ids"].shape[-1]
        content = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)

        clean_content = re.sub(r'.*</think>\s*', '', content, flags=re.DOTALL).strip()
        results[model_info["name"]] = clean_content
        print(f"   ✅ الرد: '{clean_content}'\n")

    except Exception as e:
        print(f"   ❌ فشل! نوع الخطأ: {type(e).__name__}")
        traceback.print_exc()
        results[model_info["name"]] = None
        print()

    finally:
        # هاد الجزء بينفّذ دايماً — نجح الموديل ولا فشل
        if model is not None:
            del model
        gc.collect()
        torch.cuda.empty_cache()

print("✅ انتهى الاختبار!\n")
print("📋 ملخص النتائج:")
for name, result in results.items():
    print(f"\n{name}:\n{result}")

GPU: Tesla T4
🔍 بدء اختبار النماذج محلياً على GPU...

⏳ جاري تحميل: Qwen/Qwen3-8B


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

   ✅ الرد: 'Name: Brew Haven | Tagline: Your Daily Dose of Caffeine and Community'

⏳ جاري تحميل: LiquidAI/LFM2.5-2.6B


Loading weights:   0%|          | 0/266 [00:00<?, ?it/s]

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


   ✅ الرد: 'The user wants me to generate a brand name and a short tagline for a modern coffee shop targeting university students. The response must be in the specific format: "Name: ... | Tagline: ..."

I need to create:
1. A catchy, modern brand name suitable for a coffee shop aimed at university students (think trendy, energetic, student-friendly)
2. A concise tagline that captures the essence of the brand

Brainstorming names:
- "Campus Brew" - classic but maybe too generic
- "Study & Sip" - directly targets students
- "The Lecture Latte" - plays on academic lectures
- "Cup of Knowledge" - educational vibe
- "Grind & Grind" - puns on grinding beans and studying hard
- "Student Bean" - straightforward
- "The Study Spot" - functional
- "Perk & Paper" - combines coffee (perk) with academic papers
- "Caffeine Campus" - direct reference to campus life
- "The Morning Shift" - appeals to early morning study sessions
- "Brewed Scholars" - academic twist
- "Latte Lab" - scientific/modern fe

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch, gc, re, json, requests
import pandas as pd

print(f"GPU: {torch.cuda.get_device_name(0)}\n")

# تحميل البريفات مباشرة من GitHub
url = "https://raw.githubusercontent.com/maram-elaian/brandora/main/data/test_briefs.json"
briefs = requests.get(url).json()
print(f"📋 عدد البريفات: {len(briefs)}\n")

def brief_to_prompt(b):
    return (
        f"Generate a brand name and a short tagline for this business.\n"
        f"Industry: {b['industry']}\n"
        f"Target audience: {b['target_audience']}\n"
        f"Brand purpose: {b['brand_purpose']}\n"
        f"Personality: {', '.join(b['personality'])}\n"
        f"Tone: {b['tone']}\n"
        f"Reply ONLY in this exact format: Name: ... | Tagline: ..."
    )

def clean_output(text):
    return re.sub(r'.*</think>\s*', '', text, flags=re.DOTALL).strip()

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

models_to_check = [
    {"name": "Qwen3-8B", "model_id": "Qwen/Qwen3-8B"},
    {"name": "LFM2.5-2.6B", "model_id": "LiquidAI/LFM2.5-2.6B"},
]

all_results = []

for model_info in models_to_check:
    print(f"⏳ جاري تحميل: {model_info['name']}")
    model = None
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_info["model_id"])
        model = AutoModelForCausalLM.from_pretrained(
            model_info["model_id"],
            quantization_config=quant_config,
            device_map="auto"
        )

        for i, brief in enumerate(briefs):
            messages = [{"role": "user", "content": brief_to_prompt(brief)}]

            try:
                inputs = tokenizer.apply_chat_template(
                    messages, add_generation_prompt=True,
                    return_tensors="pt", return_dict=True,
                    enable_thinking=False
                ).to(model.device)
            except TypeError:
                inputs = tokenizer.apply_chat_template(
                    messages, add_generation_prompt=True,
                    return_tensors="pt", return_dict=True
                ).to(model.device)

            outputs = model.generate(**inputs, max_new_tokens=150)
            input_length = inputs["input_ids"].shape[-1]
            raw = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
            clean = clean_output(raw)

            all_results.append({
                "brief_id": brief["id"],
                "industry": brief["industry"],
                "model": model_info["name"],
                "output": clean
            })

            print(f"   [{i+1}/{len(briefs)}] {brief['id']} ✅")

    except Exception as e:
        print(f"   ❌ فشل الموديل كله! {type(e).__name__}: {e}")

    finally:
        if model is not None:
            del model
        gc.collect()
        torch.cuda.empty_cache()

    print()

# حفظ النتائج بجدول عشان تقارنيهم بسهولة
df = pd.DataFrame(all_results)
df.to_csv("comparison_results.csv", index=False)
print("✅ خلصت! النتائج محفوظة بـ comparison_results.csv")
df.head(10)

GPU: Tesla T4

📋 عدد البريفات: 30

⏳ جاري تحميل: Qwen3-8B


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

   [1/30] BR001 ✅
   [2/30] BR002 ✅
   [3/30] BR003 ✅
   [4/30] BR004 ✅
   [5/30] BR005 ✅
   [6/30] BR006 ✅
   [7/30] BR007 ✅
   [8/30] BR008 ✅
   [9/30] BR009 ✅
   [10/30] BR010 ✅
   [11/30] BR011 ✅
   [12/30] BR012 ✅
   [13/30] BR013 ✅
   [14/30] BR014 ✅
   [15/30] BR015 ✅
   [16/30] BR016 ✅
   [17/30] BR017 ✅
   [18/30] BR018 ✅
   [19/30] BR019 ✅
   [20/30] BR020 ✅
   [21/30] BR021 ✅
   [22/30] BR022 ✅
   [23/30] BR023 ✅
   [24/30] BR024 ✅
   [25/30] BR025 ✅
   [26/30] BR026 ✅
   [27/30] BR027 ✅
   [28/30] BR028 ✅
   [29/30] BR029 ✅
   [30/30] BR030 ✅

⏳ جاري تحميل: LFM2.5-2.6B


Loading weights:   0%|          | 0/266 [00:00<?, ?it/s]

   [1/30] BR001 ✅
   [2/30] BR002 ✅
   [3/30] BR003 ✅
   [4/30] BR004 ✅
   [5/30] BR005 ✅
   [6/30] BR006 ✅
   [7/30] BR007 ✅
   [8/30] BR008 ✅
   [9/30] BR009 ✅
   [10/30] BR010 ✅
